### POPISNÉ MODELY PRE ANALÝZU DÁT O PACIENTOCH S COVID-19

#### Štvrtá vlna

Autor: Bc. Šimon Mikolaj

#### 1. Pochopenie a príprava dát

In [ ]:
import pandas as pd
import numpy as np
import re

# Načítanie Excelového súboru 
file_path = "4. vlna všetko 28-11-2024.xlsx"
df = pd.read_excel(file_path)

# Prehľad chýbajúcich hodnôt v PÔVODNOM datasete 
orig_missing_counts = df.isna().sum()
orig_missing_percent = (orig_missing_counts / len(df)) * 100

orig_missing_table = pd.DataFrame({
    "Chýbajúce hodnoty": orig_missing_counts,
    "Chýbajúce hodnoty (%)": orig_missing_percent.round(2)
}).sort_values(by="Chýbajúce hodnoty (%)", ascending=False)

print("\nPrehľad chýbajúcich hodnôt v PÔVODNOM datasete:\n")
print(orig_missing_table)

#  Identifikácia skupín atribútov laboratórnych testov
pattern = re.compile(r'(.+)\s+(first|last|min|max)$', re.IGNORECASE)
grouped = {}

for col in df.columns:
    match = pattern.match(str(col).strip())
    if match:
        base_name = match.group(1).strip()
        suffix = match.group(2).lower()
        grouped.setdefault(base_name, []).append(suffix)

# Určenie, ktoré sú laboratórne skupiny - first,last,min,max
bio_groups = {base for base, suffixes in grouped.items() if len(suffixes) > 1}

# Výber stĺpcov 
cols_to_keep = []
for col in df.columns:
    match = pattern.match(str(col).strip())
    if match:
        base_name = match.group(1).strip()
        suffix = match.group(2).lower()
        # Ak ide o laboratórny test, nechaj len 'last'
        if base_name in bio_groups and suffix == 'last':
            cols_to_keep.append(col)
    else:
        # Nejde o laboratórny atribút → nechaj ho
        cols_to_keep.append(col)

df = df[cols_to_keep]

# Odstránenie všetkých stĺpcov s veľkým počtom chýbajúcich hodnôt - podľa analýzy literatúry nad 25% a odporúčania lekárov
remove_prefixes = ["Typ vakcíny", "S-IgA", "S-Chol", "S-Ig M", "S-IgG", "S-AMS", "S-LD", "S-KM"]
cols_to_drop = [
    col for col in df.columns 
    if any(col.strip().lower().startswith(prefix.lower()) for prefix in remove_prefixes)
]
df = df.drop(columns=cols_to_drop, errors='ignore')

# Odstránenie atribútu "Poradie"
if "Poradie" in df.columns:
    df = df.drop(columns=["Poradie"])

# Odstránenie atribútov od 3. po 22. stĺpec (vrátane) - atributy s klinickým popisom
if len(df.columns) >= 22:
    cols_to_drop = df.columns[3:22]  # indexy 3 až 21
    df = df.drop(columns=cols_to_drop)

df = df.dropna(axis=1, how='all')               # Odstránenie prázdnych stĺpcov 
df = df.replace({True: 1, False: 0})            # TRUE/FALSE → 1/0 

# Spracovanie atribútu "Eo abs"
if "Eo abs last" in df.columns:
    df["Eo abs last"] = pd.to_numeric(df["Eo abs last"], errors='coerce')
    df["Eo abs last"] = df["Eo abs last"].replace(0, np.nan)


# Odstránenie stĺpcov s > 25 % chýbajúcich hodnôt 
threshold = 25.0  # percentá

missing_counts = df.isna().sum()
missing_percent = (missing_counts / len(df)) * 100
cols_over_threshold = missing_percent[missing_percent > threshold].index.tolist()

if cols_over_threshold:
    print(f"\nStĺpce s viac ako {threshold}% chýbajúcich hodnôt (budú odstránené):")
    for col in cols_over_threshold:
        print(f"  - {col}: {missing_percent[col]:.2f} %")
    df = df.drop(columns=cols_over_threshold)
else:
    print(f"\nŽiadne stĺpce nemajú viac ako {threshold}% chýbajúcich hodnôt.")

# Výpočet chýbajúcich hodnôt 
missing_counts_final = df.isna().sum()
missing_percent_final = (missing_counts_final / len(df)) * 100
missing_table = pd.DataFrame({
    'Chýbajúce hodnoty': missing_counts_final,
    'Chýbajúce hodnoty (%)': missing_percent_final.round(2)
}).sort_values(by='Chýbajúce hodnoty (%)', ascending=False)

print("\nPrehľad chýbajúcich hodnôt po odstránení stĺpcov s > 25 % chýbania:\n")
print(missing_table)

In [ ]:
# Odstránenie liekových atribútov od "MD652 | FABIFLU TABLETS" po "ANOPYRIN" 
start_col = "MD652 | FABIFLU TABLETS"
end_col = "ANOPYRIN"

if start_col in df.columns and end_col in df.columns:
    start_idx = df.columns.get_loc(start_col)
    end_idx = df.columns.get_loc(end_col)
    if start_idx <= end_idx:
        cols_to_drop = df.columns[start_idx:end_idx + 1]
        df = df.drop(columns=cols_to_drop)

In [ ]:
# Výpočet minima a maxima pre každý číselný atribút 
# Vyber len číselné stĺpce
numeric_cols = df.select_dtypes(include=[np.number])

# Vytvor prehľad s min a max
range_table = pd.DataFrame({
    'Min': numeric_cols.min(),
    'Max': numeric_cols.max()
}).sort_index()

print("Minimálne a maximálne hodnoty pre každý číselný atribút:\n")
print(range_table)

In [ ]:
# Absolútny a relatívny počet chýbajúcich atribútov
def missing_overall(df):
    total_missing = df.isna().sum().sum()
    total_cells = df.shape[0] * df.shape[1]
    percent = (total_missing / total_cells) * 100
    print("Celkový počet chýbajúcich hodnôt:", total_missing)
    print(f"Percento chýbajúcich hodnôt v celom DataFrame: {percent:.2f}%")
    return total_missing, percent

missing_overall(df)

In [ ]:
# Nahradenie nereálnych hodnôt NaN hodnotou

# Povoleny rozsah hodnôt - hranice sú od lekárov (min, max)
bounds = {
    "APTT-R last": (0.2, 8),
    "CD19+ last": (0, 5),
    "CD3+ last": (0, 5),
    "CD4+ last": (0, 5),
    "CD4+/CD8+ last": (0, 5),
    "CD8+ last": (0, 5),
    "D-dimér HS last": (0, 10),
    "Eo abs last": (0, 10),
    "Fib last": (0, 10),
    "HGB last": (0, 22),
    "Ly abs last": (0, 15),
    "NE/LY(NLR) last": (0, 50),
    "NK last": (0, 5),
    "Neu abs last": (0, 50),
    "P-Laktát last": (0, 10),
    "PDW last": (0, 50),
    "PLT last": (0, 3000),
    "PT (INR) last": (0.2, 8),
    "S-ALP last": (0.1, 50),
    "S-ALT last": (0.1, 150),
    "S-AST last": (0.1, 150),
    "S-Alb last": (10, 100),
    "S-Bil-T last": (1, 600),
    "S-CB last": (10, 150),
    "S-CK last": (0.1, 50),
    "S-CK-MB last": (0, 20),
    "S-CL last": (70, 150),
    "S-CRP last": (0, 2000),
    "S-FER last": (1, 10000),
    "S-GMT last": (0.1, 150),
    "S-Gluk last": (0.1, 50),
    "S-IL6 last": (0.1, 10000),
    "S-K last": (1, 10),
    "S-Kreat last": (10, 2000),
    "S-Na last": (100, 200),
    "S-TnT last": (0, 100),
    "S-PBNP last": (0, 50000),
    "S-VITD last": (0, 1000),
    "SatO2 %": (30, 100),
    "WBC last": (0.1, 60),
}

def apply_medical_bounds(df, bounds_dict):
    report = []

    for col, (min_val, max_val) in bounds_dict.items():
        if col not in df.columns:
            print(f"Atribút '{col}' sa v datasete nenašiel.")
            continue

        # konverzia na numeric
        df[col] = pd.to_numeric(df[col], errors="coerce")

        # detekcia nereálnych hodnôt
        invalid_mask = (df[col] < min_val) | (df[col] > max_val)
        invalid_count = invalid_mask.sum()

        # nahradenie NaN
        df.loc[invalid_mask, col] = np.nan

        # uloženie
        report.append({
            "atribút": col,
            "min": min_val,
            "max": max_val,
            "nahradené_hodnoty": int(invalid_count)
        })

        if invalid_count > 0:
            print(f" {col}: nahradených {invalid_count} nereálnych hodnôt")

    report_df = pd.DataFrame(report)
    return df, report_df


df, bounds_report = apply_medical_bounds(df, bounds)

print("\n Súhrn čistenia:")
print(bounds_report.sort_values("nahradené_hodnoty", ascending=False))

In [ ]:
# Odstráň pacientov ktorí majú viac ako 40% chýbajucich hodnôt - na základe analýzy literatúry

threshold = 0.4                                                             
missing_ratio = df.isna().mean(axis=1)                                  # Podiel chýbajúcich hodnôt v každom riadku
odstranene_idx = missing_ratio[missing_ratio > threshold].index         # Indexy pacientov, ktorí budú odstránení
odstranene_mena = df.loc[odstranene_idx, "Meno"]                        # Mená (zašifrované) pacientov, ktorí budú odstránení
pocet_odstranenych = len(odstranene_idx)
print("Odstránení pacienti (viac ako 40 % chýbajúcich hodnôt):")
print(odstranene_mena.to_string(index=False))

df = df[missing_ratio <= threshold]                                     # Odstránenie pacientov
print(f"\nOdstránených {pocet_odstranenych} pacientov.")
print(f"Zostalo {len(df)} pacientov.")

In [ ]:
# Odstránenie pacientov, ktorí nemajú cieľový atribút - Závažnosť priebehu ochorenia

missing_target_idx = df[df["Závažnosť priebehu ochorenia"].isna()].index
    
if len(missing_target_idx) > 0:
    missing_target_names = df.loc[missing_target_idx, "Meno"]
    print("\nPacienti odstránení kvôli chýbajúcej hodnote Závažnosť priebehu ochorenia:")
    print(missing_target_names.to_string(index=False))
        
    df = df.drop(index=missing_target_idx)                                                                  # odstránenie
    print(f"\nOdstránených {len(missing_target_idx)} pacientov kvôli chýbajúcej cieľovej hodnote.")
else:
    print("\nŽiadny pacient nemá chýbajúcu hodnotu v 'Závažnosť priebehu ochorenia'.")

print(f"Aktuálny počet pacientov po odstránení: {len(df)}")

In [ ]:
# Zistenie typu chýbajúcich hodnôt na základe korelačnej matice

import seaborn as sns
import matplotlib.pyplot as plt

missing_mask = df.isna()                                    # Maska chýbajúcich hodnôt
corr_missing = missing_mask.corr()                          # Korelačná matica medzi maskou chýbajúcich hodnôt

plt.figure(figsize=(15,12))
sns.heatmap(corr_missing, cmap="coolwarm", center=0)
plt.title("Korelácie medzi chýbaniami premenných")
plt.show()

In [ ]:
# Výpis vysoko korelovaných atribútov z hľadiska chýbajúcich atribútov

high_corr = corr_missing[(corr_missing > 0.3) & (corr_missing < 1.0)].stack().sort_values(ascending=False)
print("Silnejšie závislosti medzi chýbaniami hodnôt:\n", high_corr.head(30))

In [ ]:
# Iná vizualizácia chýbajúcich hodnôt

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(14,12))
sns.heatmap(df.isna(), cbar=False)
plt.title("Vizualizácia chýbajúcich hodnôt")
plt.show()

In [ ]:
# MICE Imputácia

import miceforest as mf

df["Pohlavie"] = df["Pohlavie"].astype("category")                          # konverzia pohlavia na kategorický atribút
df["Pohlavie_code"] = df["Pohlavie"].cat.codes
df = df.reset_index(drop=True)

cols_to_exclude = ["Meno", "Závažnosť priebehu ochorenia", "Pohlavie"]      # odstánenie atribútov, ktoré nechceme imputovať 
X = df.drop(columns=cols_to_exclude)

X = X.reset_index(drop=True)                                                # reset indexu (nutné pre miceforest)
kernel = mf.ImputationKernel(                                               # vytvorenie MICE karenlu – minimal api
    X,
    random_state=42
)

kernel.mice(iterations=30, verbose=True)                                    # spustenie imputácie s 30 iteráciami
X_imputed = kernel.complete_data()                                          # doplnené dáta

for c in cols_to_exclude:                                                   # vrátenie neimputovaných atribútov späť
    X_imputed[c] = df[c]

print("MICE miceforest imputácia úspešne dokončená!")

In [ ]:
# Uložím imputovaný dataset do Excelu
X_imputed.to_excel("imputed_dataset.xlsx", index=False)
# aby som mal df po MICE vzdy
X_imputed = pd.read_excel("imputed_dataset.xlsx")

In [ ]:
# Výpis počtu pacientov podľa cieľového atribútu
print(X_imputed["Závažnosť priebehu ochorenia"].value_counts(dropna=False))

# možnosť porovania distribúcie hodnôt konkrétneho atribútu pred a po imputácii 
def compare_distributions(original, imputed, col):
    plt.figure(figsize=(6,4))
    original[col].dropna().plot(kind="kde", label="Original", linewidth=2)
    imputed[col].plot(kind="kde", label="Imputed", linewidth=2, alpha=0.7)
    plt.title(f"Distribúcia pre {col}")
    plt.legend()
    plt.show()

compare_distributions(df, X_imputed, "S-Alb last")

In [ ]:
# Odstránenie korelovaných atribútov nad 85%, ignoruje binárne

import numpy as np
import pandas as pd

def correlation_filter_with_report_no_binary(df, threshold=0.95):

    # nájdeme binárne premenné (0/1)
    binary_cols = [
        col for col in df.columns
        if set(df[col].dropna().unique()).issubset({0, 1})
    ]

    print("Binarne premenné ignorované v korelácii:", binary_cols)

    continuous_cols = [col for col in df.columns if col not in binary_cols]             # nebinárne numerické atribúty
    corr_matrix = df[continuous_cols].corr().abs()                                      # korelačná matica len z nebinárnych atribútov
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))    # horný trojuholník korelácie

    removed_variables = []
    removal_report = []

    # hľadaj vysoko korelované atribúty
    for col in upper.columns:
        high_corr = upper.index[upper[col] > threshold].tolist()

        if len(high_corr) > 0:
            for correlated_with in high_corr:
                removal_report.append({
                    "removed_variable": col,
                    "correlated_with": correlated_with,
                    "correlation_value": corr_matrix.loc[col, correlated_with]
                })
            removed_variables.append(col)

    df_filtered = df.drop(columns=removed_variables, errors='ignore')                   # odstránenie korelovaných atribútov
    report_df = pd.DataFrame(removal_report)                                            # report

    return df_filtered, report_df


# ---- Použitie ----
# Odstránim tie, ktoré nemajú ísť do numerickej korelácie:
X_numeric = X_imputed.drop(columns=["Závažnosť priebehu ochorenia", "Meno", "Pohlavie"])

X_filtered, corr_report = correlation_filter_with_report_no_binary(X_numeric, threshold=0.85)

print("\nOdstránené atribúty:")
print(corr_report["removed_variable"].unique())

print("\nDETAILNÝ REPORT:")
print(corr_report)

In [ ]:
# Výber najdôležitejších atribútov vzhľadom k cieľovému atribútu pomocou EXTRA TREES Classifier

from sklearn.ensemble import ExtraTreesClassifier

y = X_imputed["Závažnosť priebehu ochorenia"]       # y = cielový atribút
X_feat = X_imputed[X_filtered.columns]              # X = imputované dáta po korelacii(numerické + Pohlavie_code)

# Extra Trees model
model = ExtraTreesClassifier(n_estimators=300, random_state=42)
model.fit(X_feat, y)

# Významnosť atribútov
importances = pd.Series(model.feature_importances_, index=X_feat.columns)
top_features = importances.sort_values(ascending=False)     # zoradenie atribútov od najdôležitejších

print(top_features)

In [ ]:
# výpis dôležitosti s postupným sčítavaním
cum_importance = top_features.cumsum()
print(cum_importance)

# graf dôležitosti atribútov
plt.figure(figsize=(10,8))
top_features.plot(kind="bar")

In [ ]:
### vyber naj 30 atribútov - prispievajú k rozhodovaniu Extra tree na 80%  ####ODSTRANENIE POHLAVIE CODE A REnalne ochorenia

top36 = list(top_features.index[:30])
top36 = [col for col in top36 if col != "Renálne ochorenia"]   # odstránime "Renálne ochorenia", lebo ide o binárny atribút 
# pri tvorbe algoritmu bez atribútu "P-Laktát last", ktorý bude vstupovať do aplikácie, rovnako odstránime aj tento atribút

top36 = list(dict.fromkeys(top36))  # odstráni duplicity
X_top36 = X_imputed[top36].copy()

In [ ]:
# Normalizacia - štandardizácia na nulový stred

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X_top36)                                          # Normalizované dáta (numpy array)
X_scaled = pd.DataFrame(X_scaled_array, columns=X_top36.columns, index=X_top36.index)   # Pre lepšiu prácu si to prekonvertujeme späť na DataFrame

print("Normalizácia dokončená.")
X_scaled.head()

---

#### 2. Modelovanie

In [ ]:
# Testovanie optimálneho počtu zhlukov - K-MEANS algoritmus

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt

wcss = []
silhouette_scores = []
db_scores = []

K_range = range(2, 11)                                          # otestujeme od 2 do 10 zhlukov

for k in K_range:
    # vždy nový model s iným k
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)   # 10 krát sa spustí s rôznymi počiatočnými centroidmi
    labels = kmeans.fit_predict(X_scaled)

    wcss.append(kmeans.inertia_)                                # WCSS (Within-Cluster Sum of Squares) z k-means
    sil_score = silhouette_score(X_scaled, labels)              # Silhouette skóre
    silhouette_scores.append(sil_score)
    db_scores.append(davies_bouldin_score(X_scaled, labels))    # Davies–Bouldin index  

# vizualizácia výsledkov
plt.figure(figsize=(16,5))

plt.subplot(1,3,1)
plt.plot(K_range, wcss, marker='o')
plt.title("WCSS (lakťová metóda) — k-means")
plt.xlabel("Počet zhlukov (k)")
plt.ylabel("WCSS")

plt.subplot(1,3,2)
plt.plot(K_range, silhouette_scores, marker='o')
plt.title("Silhouette skóre — k-means")
plt.xlabel("Počet zhlukov (k)")
plt.ylabel("Silhouette")

plt.subplot(1,3,3)
plt.plot(K_range, db_scores, marker='o')
plt.title("Davies–Bouldin index")
plt.xlabel("Počet zhlukov (k)")
plt.ylabel("DB index (nižšie je lepšie)")

plt.tight_layout()
plt.show()

In [ ]:
# Silhouette pásikový graf pre K-MEANS

import matplotlib.cm as cm
from sklearn.metrics import silhouette_samples, silhouette_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

range_n_clusters = K_range   

for n_clusters in range_n_clusters:
    # vytvoríme jeden obrázok s jedným subplotom (len silhouette plot)
    fig, ax1 = plt.subplots(1, 1)
    fig.set_size_inches(6, 4)

    # Silhouette hodnota je v rozsahu [-1, 1]
    ax1.set_xlim([-0.1, 1])
    # Výška grafu = počet vzoriek + medzery medzi zhlukmi
    ax1.set_ylim([0, len(X_scaled) + (n_clusters + 1) * 10])
    # KMeans model
    clusterer = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = clusterer.fit_predict(X_scaled)
    # Priemerné silhouette skóre
    silhouette_avg = silhouette_score(X_scaled, cluster_labels)
    print(f"Pre k = {n_clusters} je priemerné silhouette skóre: {silhouette_avg:.3f}")
    # Silhouette skóre pre každý jeden bod
    sample_silhouette_values = silhouette_samples(X_scaled, cluster_labels)

    y_lower = 10
    for i in range(n_clusters):
        # silhouette pre body patriace do zhluku i
        ith_cluster_sil_values = sample_silhouette_values[cluster_labels == i]
        ith_cluster_sil_values.sort()

        size_cluster_i = ith_cluster_sil_values.shape[0]
        y_upper = y_lower + size_cluster_i

        color = cm.nipy_spectral(float(i) / n_clusters)
        ax1.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            ith_cluster_sil_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7,
        )

        # označenie zhluku do stredu pásu
        ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        # posun pre ďalší zhluk
        y_lower = y_upper + 10

    # vizualizácia
    ax1.set_title(f"Silhouette plot pre k = {n_clusters} (k-means)")
    ax1.set_xlabel("Silhouette koeficient")
    ax1.set_ylabel("Označenie zhluku")

    ax1.axvline(x=silhouette_avg, color="red", linestyle="--")                      # zvislá čiara – priemerné silhouette skóre
    ax1.set_yticks([])  # na osi Y nechceme menovky
    ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

    plt.tight_layout()
    plt.show()

In [ ]:
### ZHLUKOVANIE K-MEANS

from sklearn.cluster import KMeans

k = 3
model = KMeans(n_clusters=k, random_state=42, n_init=10)
cluster_labels = model.fit_predict(X_scaled)

# pridáme zhluky do pôvodného imputovaného datasetu
X_imputed["cluster"] = cluster_labels
centroids = model.cluster_centers_
print("Zhlukovanie dokončené.")

---

Podobne ako pri K-medoids vykonáme testovanie optimálneho počtu zhlukov, pasikový Silhouette index a samotné zhlukovanie aj pri zhlukovaní K-medoids

In [ ]:
# Testovanie optimálaneho počtu zhlukov K-MEDOIDS

import numpy as np
from pyclustering.cluster.kmedoids import kmedoids
from sklearn.metrics import silhouette_score
from pyclustering.utils.metric import distance_metric, type_metric

def run_kmedoids(X_scaled, k, random_state=42, n_init=10):          # 10 krát sa spustí s rôznymi počiatočnými medoidmi
    rng = np.random.RandomState(random_state)

    best_labels = None
    best_medoids = None
    best_sil = -1

    X_np = X_scaled.values                                          # normailizované hodnoty

    for i in range(n_init):
        init_medoids = rng.choice(len(X_np), k, replace=False)      # náhodný výber počiatočných medoidov

        kmed = kmedoids(
            data=X_np,
            initial_index_medoids=init_medoids,
            metric=distance_metric(type_metric.EUCLIDEAN)           # euklidovská vzdialenosť
        )
        kmed.process()                                              # samotné zhlukovanie
        clusters = kmed.get_clusters()                              # zoznam zhlukov
        medoids = kmed.get_medoids()                                # medoidy

        labels = np.zeros(len(X_np), dtype=int)                     # každému bodu sa priradí číslo zhluku
        for cid, pts in enumerate(clusters):
            for p in pts:
                labels[p] = cid

        sil = silhouette_score(X_np, labels)                        # Silhouette index výpočet

        if sil > best_sil:
            best_sil = sil
            best_labels = labels
            best_medoids = medoids

    return best_labels, best_medoids, best_sil                      # vracia najlepšie priradenie bodov do zhlukov, meedoidy aj silhouette skore


from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt

def evaluate_kmedoids(X_scaled, K_range=range(2, 11)):              # otestujeme od 2 do 10 zhlukov

    wcss = []
    sil_scores = []
    db_scores = []

    for k in K_range:
        labels, medoids, sil = run_kmedoids(
            X_scaled,
            k=k,
            random_state=42,
            n_init=10
        )

        # "WCSS" – suma vzdialeností k medoidom, nie je to presne km.inertia_ ako v k-means, tu treba dopočítať
        wcss_k = 0.0
        for cid, med_idx in enumerate(medoids):
            cluster_points = np.where(labels == cid)[0]
            dists = np.linalg.norm(
                X_scaled.values[cluster_points] - X_scaled.values[med_idx],
                axis=1
            )
            wcss_k += np.sum(dists ** 2)
        wcss.append(wcss_k)
        
        sil_scores.append(silhouette_score(X_scaled, labels))                       # Silhouette skóre
        db_scores.append(davies_bouldin_score(X_scaled, labels))                    # Davies–Bouldin index

    # vizualizácia
    plt.figure(figsize=(16, 5))

    plt.subplot(1, 3, 1)
    plt.plot(K_range, wcss, marker="o")
    plt.title("WCSS (lakťová metóda) — k-medoids")
    plt.xlabel("Počet zhlukov (k)")
    plt.ylabel("WCSS")

    plt.subplot(1, 3, 2)
    plt.plot(K_range, sil_scores, marker="o")
    plt.title("Silhouette skóre — k-medoids")
    plt.xlabel("Počet zhlukov (k)")
    plt.ylabel("Silhouette")

    plt.subplot(1, 3, 3)
    plt.plot(K_range, db_scores, marker="o")
    plt.title("Davies–Bouldin index — k-medoids")
    plt.xlabel("Počet zhlukov (k)")
    plt.ylabel("DB index (nižšie je lepšie)")

    plt.tight_layout()
    plt.show()

    return wcss, sil_scores, db_scores

# použitie:
wcss_kmed, sil_kmed, db_kmed = evaluate_kmedoids(X_scaled, K_range=range(2, 11))

In [ ]:
# Silhouette pásikový graf pre K-MEDOIDS

from sklearn.metrics import silhouette_samples
import matplotlib.cm as cm

def silhouette_plot_kmedoids(X_scaled, K_range=range(2, 7)):

    for n_clusters in K_range:
        labels, medoids, sil_avg = run_kmedoids(
            X_scaled,
            k=n_clusters,
            random_state=42,
            n_init=10
        )

        fig, ax1 = plt.subplots(figsize=(7,5))
        fig.set_size_inches(6, 4)
        ax1.set_xlim([-0.1, 1])
        ax1.set_ylim([0, len(X_scaled) + (k + 1) * 10])

        sample_sil = silhouette_samples(X_scaled, labels)
        sil_avg = np.mean(sample_sil)
        print(f"Pre k = {n_clusters} je priemerné silhouette skóre: {sil_avg:.3f}")


        y_lower = 10
        for i in range(n_clusters):
            vals = sample_sil[labels == i]
            vals.sort()

            size_i = vals.shape[0]
            y_upper = y_lower + size_i

            color = cm.nipy_spectral(float(i) / n_clusters)
            ax1.fill_betweenx(np.arange(y_lower, y_upper), 0, vals,
                            facecolor=color, edgecolor=color, alpha=0.7)

            ax1.text(-0.05, y_lower + 0.5 * size_i, str(i))
            y_lower = y_upper + 10

        ax1.axvline(x=sil_avg, color="red", linestyle="--")
        ax1.set_title(f"Silhouette graf pre k = {n_clusters} (k-medoids)")
        ax1.set_xlabel("Silhouette skóre")
        ax1.set_ylabel("Označenie zhluku")
        ax1.set_yticks([])
        ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

        plt.tight_layout()
        plt.show()

silhouette_plot_kmedoids(X_scaled, K_range=range(2, 7))

In [ ]:
### ZHLUKOVANIE k-medoids

k = 3
cluster_labels, medoids, sil = run_kmedoids(
    X_scaled,
    k=k,
    random_state=42,
    n_init=10
)

# Pridáme výsledky do pôvodného datasetu
X_imputed["cluster"] = cluster_labels

print("Zhlukovanie dokončené pomocou stabilného K-Medoids")
print(f"Medoidy: {medoids}")
print(f"Silhouette: {sil:.3f}")

---

Znovu podobný postup testovania optimálneho počtu zhlukov a vizualizácia Silhouette pásikového indexu, tentokrát pre Aglomeratívne zhlukovanie.

In [ ]:
# Testovanie optimálaneho počtu zhlukov Aglomerativneho zhlukovania

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
import numpy as np

def evaluate_agglomerative(X_scaled, K_range=range(2, 11), linkage="ward"):     # otestujeme od 2 do 10 zhlukov, použijeme Wardovú metódu agl. zhlukovania
    wcss = []
    sil_scores = []
    db_scores = []

    X_np = X_scaled.values

    for k in K_range:
        hc = AgglomerativeClustering(
            n_clusters=k,
            linkage=linkage
        )
        labels = hc.fit_predict(X_np)

        # WCSS – spočítame ručne cez centroidy zhlukov
        wcss_k = 0.0
        for cluster_id in np.unique(labels):
            cluster_points = X_np[labels == cluster_id]
            centroid = cluster_points.mean(axis=0)
            wcss_k += ((cluster_points - centroid) ** 2).sum()
        wcss.append(wcss_k)

        sil_scores.append(silhouette_score(X_np, labels))                       # Silhouette skóre
        db_scores.append(davies_bouldin_score(X_np, labels))                    # Davies–Bouldin index           

    # Vizualizácia 
    plt.figure(figsize=(16, 5))

    plt.subplot(1, 3, 1)
    plt.plot(K_range, wcss, marker="o")
    plt.title(f"WCSS (lakťová metóda) — agglomerative")    
    plt.xlabel("Počet zhlukov (k)")
    plt.ylabel("WCSS")

    plt.subplot(1, 3, 2)
    plt.plot(K_range, sil_scores, marker="o")
    plt.title(f"Silhouette skóre — agglomerative")
    plt.xlabel("Počet zhlukov (k)")
    plt.ylabel("Silhouette")

    plt.subplot(1, 3, 3)
    plt.plot(K_range, db_scores, marker="o")
    plt.title(f"Davies–Bouldin index — agglomerative")
    plt.xlabel("Počet zhlukov (k)")
    plt.ylabel("DB index (nižšie je lepšie)")

    plt.tight_layout()
    plt.show()

    return wcss, sil_scores, db_scores

# použitie:
wcss_hc, sil_hc, db_hc = evaluate_agglomerative(X_scaled, K_range=range(2, 11), linkage="ward")

In [ ]:
# Silhouette pásikový graf pre AGLOMERATIVNE zhlukovanie

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_samples, silhouette_score

def silhouette_barplot_agglomerative(X_scaled, K_range=range(2, 11), linkage="ward"):
    X_np = X_scaled.values

    for n_clusters in K_range:
        # --- Aglomeratívne zhlukovanie 
        # pri linkage="ward" musí byť metric="euclidean"
        if linkage == "ward":
            model = AgglomerativeClustering(n_clusters=n_clusters, linkage="ward", metric="euclidean")
        else:
            model = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage, metric="euclidean")

        labels = model.fit_predict(X_np)

        # --- Silhouette metriky ---
        silhouette_avg = silhouette_score(X_np, labels)
        sample_silhouette_values = silhouette_samples(X_np, labels)

        # --- Kreslenie pásikového silhouette grafu ---
        fig, ax1 = plt.subplots(1, 1)
        fig.set_size_inches(6, 4)

        ax1.set_xlim([-0.1, 1])
        ax1.set_ylim([0, len(X_np) + (n_clusters + 1) * 10])

        y_lower = 10
        for i in range(n_clusters):
            ith_cluster_sil_values = sample_silhouette_values[labels == i]
            ith_cluster_sil_values.sort()

            size_cluster_i = ith_cluster_sil_values.shape[0]
            y_upper = y_lower + size_cluster_i

            color = cm.nipy_spectral(float(i) / n_clusters)
            ax1.fill_betweenx(
                np.arange(y_lower, y_upper),
                0,
                ith_cluster_sil_values,
                facecolor=color,
                edgecolor=color,
                alpha=0.7,
            )

            ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
            y_lower = y_upper + 10

            ax1.set_title(f"Silhouette graf pre k = {n_clusters} (agglomerative)")
        ax1.set_xlabel("Silhouette skóre")
        ax1.set_ylabel("Označenie zhluku")
        ax1.axvline(x=silhouette_avg, color="red", linestyle="--")
        ax1.set_yticks([])
        ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

        plt.tight_layout()
        plt.show()
        print(f"k = {n_clusters}, priemerné silhouette (agglomerative-{linkage}): {silhouette_avg:.3f}")

silhouette_barplot_agglomerative(X_scaled, K_range=range(2, 7), linkage="ward")

In [ ]:
### ZHLUKOVANIE hierarchické - aglomerativne

from sklearn.cluster import AgglomerativeClustering

k = 3                             # Počet zhlukov

# Hierarchické (agglomerative) zhlukovanie
hc_model = AgglomerativeClustering(
    n_clusters=k,
    linkage="ward"                # wardová metóda
)

cluster_labels = hc_model.fit_predict(X_scaled)
X_imputed["cluster"] = cluster_labels

print("Hierarchické zhlukovanie (AgglomerativeClustering) dokončené.")
print("Počty pacientov v zhlukoch:")
print(X_imputed["cluster"].value_counts())

In [ ]:
#  Dendrogram aglomeratívneho zhlukovania

from scipy.cluster.hierarchy import linkage, dendrogram

np.random.seed(42)
plt.figure(figsize=(9, 5))

Z = linkage(X_scaled, method='ward')
cut_height = Z[-2, 2]                                                           # berieme predposledné spojenie, to zodpovedá rozdeleniu na 3 zhluky
dendrogram(Z, truncate_mode="lastp", p=30, color_threshold=cut_height)          # zobrazí posledných 30 zhlukov v dendrograme

plt.axhline(
    y=cut_height,
    color="red",
    linestyle="--",
    linewidth=1,
    label="Rez na 3 zhluky"
)

# vizualizácia
plt.title("Dendrogram aglomeratívneho zhlukovania 4. vlny")
plt.xlabel("Pacienti (počet alebo index)")
plt.ylabel("Vzdialenosť")
plt.tight_layout()
plt.show()

---

Tu skončili zlgoritmy zhlukovania, nasleduje PCA vizualizácia, výpočet čistoty zhlukov, nasadenie štatistických testov a SHAP vizualizácia

In [ ]:
### PCA vizualizácia

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)                   # 2 hlavné komponenty
coords = pca.fit_transform(X_scaled)        # použujeme normalizované dáta

pca_df = pd.DataFrame(coords, columns=["PC1", "PC2"])           # uložíme si do DataFrame pre jednoduchší plotting

pca_df["cluster"] = cluster_labels                              # pridanie index zhluku
pca_df["target"] = X_imputed["Závažnosť priebehu ochorenia"]    # pridanie pôvodného cieľového atribútu pre porovnanie 

In [ ]:
# PCA vizualizácia podla zhlukov

plt.figure(figsize=(7,5))
plt.scatter(pca_df["PC1"], pca_df["PC2"], c=pca_df["cluster"], cmap="viridis")
plt.title("PCA vizualizácia k-medoids (k=3)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
# PCA vizualizáca podľa pôvodného cieľového atribútu

import matplotlib.patches as mpatches

plt.figure(figsize=(7,5))
scatter = plt.scatter(pca_df["PC1"], pca_df["PC2"], c=pca_df["target"], cmap="plasma")

plt.title("PCA vizualizácia — podľa pôvodnej závažnosti priebehu ochorenia")
plt.xlabel("PC1")
plt.ylabel("PC2")

# Vytvorenie legendy
unique_targets = sorted(pca_df["target"].unique())
colors = scatter.cmap(scatter.norm(unique_targets))

legend_handles = [
    mpatches.Patch(color=colors[i], label=f" {unique_targets[i]}")
    for i in range(len(unique_targets))
]

plt.legend(handles=legend_handles, title="Priebeh ochorenia")
plt.show()

In [ ]:
# Kontingenčná tabuľka závažnosť ochorenia/zhluk

ct = pd.crosstab(X_imputed["cluster"], X_imputed["Závažnosť priebehu ochorenia"])
print(ct)

### HEAT mapa
plt.figure(figsize=(6,4))
sns.heatmap(ct, annot=True, fmt="d", cmap="YlOrRd")
plt.title("Heatmapa — prekrývanie zhlukov a závažnosti")
plt.xlabel("Závažnosť priebehu ochorenia")
plt.ylabel("Zhluk")
plt.show()

In [ ]:
# Mapovanie zhlukov podľa závažnosti ochorenia

cluster_translation = ct.idxmax(axis=1)
print(cluster_translation)

In [ ]:
# Cluster purity - Čistota zhlukov

correct = 0
total = len(X_imputed)
for cl in ct.index:
    correct += ct.loc[cl].max()

purity = correct / total
print("Purity clusteringu:", round(purity, 3))

In [ ]:
# Mediány atribútov podľa clusteru 

from matplotlib.colors import LogNorm

cluster_medians = X_imputed.groupby("cluster")[top36 + ["Pohlavie_code"] + ["Renálne ochorenia"]].median()      # až tu pridávam atribút Pohlavia a Renálnych ochorení, do modelovania nevstupoval
cluster_medians_T = cluster_medians.T

# vizualizácia
plt.figure(figsize=(4,10))
sns.heatmap(
    cluster_medians_T, 
    cmap="coolwarm",
    norm=LogNorm(),        # log škála, kôli rôznemu rozsahu hodnôt
    annot=True,            # zobrazí hodnoty
    fmt=".2f"              # zaokrúhlenie
)
plt.title("Mediány atribútov podľa zhlukov")
plt.ylabel("Atribút")
plt.xlabel("Zhluk")
plt.show()

In [ ]:
# Z-score normalizácia mediánov atribútov

scaler = StandardScaler()
z_scores = scaler.fit_transform(cluster_medians_T)
z_df = pd.DataFrame(z_scores, index=cluster_medians_T.index, columns=cluster_medians_T.columns)

# vizualizácia
plt.figure(figsize=(4,10))
sns.heatmap(z_df, cmap="vlag", center=0, annot=True, fmt=".2f")
plt.title("Heatmapa atribútov (Z-score mediánov)")
plt.ylabel("Atribút")
plt.xlabel("Zhluk")
plt.show()

In [ ]:
### ŠTATISTICKÉ TESTY
    # test normality - Shapiro-Wilk test- podla toho popísanie priemeru a SD, alebo medianu a IQR atribútu
    # tiež podľa toho štatistický test - ANOVA alebo Kruskall-Wallis
    # pre kategorické atribúty - iba Pohlavie_code - Chi-square test

from scipy.stats import shapiro, f_oneway, kruskal, chi2_contingency

# atribúty, ktoré idú do porovnaia 
features_for_analysis = top36 + ["Pohlavie_code"] + ["Renálne ochorenia"] # pridávame Pohlavie a Renálne ochorenia

results = []
clusters = sorted(X_imputed["cluster"].unique())

# Loop cez všetky atribúty
for col in features_for_analysis:

    data = X_imputed[col].dropna()

    # ŠPECIÁLNY PRÍPAD: Pohlavie_code 
    if col == "Pohlavie_code":

        distribution = "Categorical (binary)"
        # Kontingenčná tabuľka cluster × pohlavie
        contingency = pd.crosstab(X_imputed["cluster"], X_imputed[col])
        chi2, p_value, dof, expected = chi2_contingency(contingency)        # Chi-square test

        # Popisné štatistiky (percentá žien/mužov)
        cluster_stats = {
            f"Zhluk {g}": 
                f"{(X_imputed[X_imputed['cluster']==g][col].mean()*100):.1f}% žien"
            for g in clusters
        }

        # uložime
        results.append({
            "Atribút": col,
            "Rozdelenie": distribution,
            **cluster_stats,
            "p_hodnota": p_value
        })

        continue  # preskočiť ďalšie testy a ísť na ďalší atribút

    #  ŠPECIÁLNY PRÍPAD: Renálne_ochorenia
    if col == "Renálne ochorenia":

        distribution = "Categorical (binary)"
        # Kontingenčná tabuľka cluster × pohlavie
        contingency = pd.crosstab(X_imputed["cluster"], X_imputed[col])
        chi2, p_value, dof, expected = chi2_contingency(contingency)        # Chi-square test

        # Popisné štatistiky (percentá žien/mužov)
        cluster_stats = {
            f"Zhluk {g}": 
                f"{(X_imputed[X_imputed['cluster']==g][col].mean()*100):.1f}%"
            for g in clusters
        }

        # uložime
        results.append({
            "Atribút": col,
            "Rozdelenie": distribution,
            **cluster_stats,
            "p_hodnota": p_value
        })

        continue  # preskočiť ďalšie testy a ísť na ďalší atribút


    #  KONTINUÁLNE PREMENNÉ - numerické (normálne / nenormálne) 
    #  Shapiro-Wilk normality test 
    sample = data.sample(min(len(data), 5000), random_state=42)
    stat_norm, p_norm = shapiro(sample)

    # Skupiny podľa clusterov 
    group_values = [X_imputed[X_imputed["cluster"] == g][col].dropna() for g in clusters]

    # Štatistický test podľa normality 
    if p_norm >= 0.05:
        # normálne rozdelenie → ANOVA test
        distribution = "Normal"
        stat_test, p_value = f_oneway(*group_values)

        # atribút popíšeme priemerom a SD 
        cluster_stats = {
            f"Zhluk {g}": 
                f"{group_values[i].mean():.2f} ± {group_values[i].std():.2f}"
            for i, g in enumerate(clusters)
        }

    else:
        # nenormálne rozdelenie → Kruskal–Wallis test
        distribution = "Non-normal"
        stat_test, p_value = kruskal(*group_values)

        # atribút popíšeme medianom a IQR
        cluster_stats = {
            f"Zhluk {g}":
                f"{group_values[i].median():.2f} [{group_values[i].quantile(0.25):.2f}; "
                f"{group_values[i].quantile(0.75):.2f}]"
            for i, g in enumerate(clusters)
        }

    # Uložíme výsledok
    results.append({
        "Atribút": col,
        "Rozdelenie": distribution,
        **cluster_stats,
        "p_hodnota": p_value
    })


# Finálna tabuľka
summary_table = pd.DataFrame(results)
summary_table = summary_table.sort_values(by="p_hodnota", ascending=True)       # zoradíme podľa p_hodnoty

# bežné zaokrúhlenie na 6 desatín
def format_p_value_strict(x):
    if pd.isna(x):
        return x
    if x < 0.00001:
        return "<0.00001"
    else:
        return f"{x:.6f}"   

# vedecký zápis
def format_p_value(x):
    if pd.isna(x):
        return x
    if x < 0.001:
        return f"{x:.2e}"   # vedecký zápis pre veľmi malé p
    else:
        return f"{x:.5f}"   # klasický zápis so 5 desatinnými miestami

summary_table["p_hodnota"] = summary_table["p_hodnota"].apply(format_p_value)
summary_table

In [ ]:
# Export do tabulky

df_export = summary_table.copy()

# Vytvorenie obrázka
fig, ax = plt.subplots(figsize=(14, len(df_export)*0.45))
ax.axis('off')

# vytvorenie tabuľky
table = ax.table(
    cellText=df_export.values,
    colLabels=df_export.columns,
    cellLoc='center',
    loc='center'
)

# Štýl bunky
table.auto_set_font_size(False)
table.set_fontsize(8.9)
table.scale(1, 2)

plt.title("Štatistické porovnanie atribútov medzi zhlukmi", fontsize=14, pad=10)
plt.savefig("cluster_stats_table.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Uloženie modelu, scaleru - štandardizácie, X_scaled - štandardizovaných dát a X_top36 - reálne hodnoty top N atribútov 
# tieto premenné budú následne použité ako vstupy pre aplikáciu
# líšia sa na základe počtu atribútov - 14 alebo 15
import joblib

joblib.dump(model, "kmeans_model_4vlna14_.pkl")
joblib.dump(scaler, "scaler_4vlna14_.pkl")
X_scaled.to_pickle("X_scaled_4vlna14.pkl")
X_top36.to_pickle("X_orig_4vlna14.pkl")

In [ ]:
# POST-HOC testy numerickych atributov

import scikit_posthocs as sp

# iba tam kde bola p-hodnota Kruskall-Vallis testu menšia ako 0.05
significant_features = summary_table.loc[
    summary_table["p_hodnota"].astype(float) < 0.05,
    "Atribút"
].tolist()

# odstranime binarny atribut Pohlavie_code a Renalne ochorania
significant_features = [f for f in significant_features if f != "Pohlavie_code"]
significant_features = [f for f in significant_features if f != "Renálne ochorenia"]
clusters = sorted(X_imputed["cluster"].unique())
posthoc_results = []

# prechádzame atribútmi
for col in significant_features:
    data = X_imputed[[col, "cluster"]].dropna()
    # Dunnov post-hoc test s Bonferroni korekciou
    dunn = sp.posthoc_dunn(
        data,
        val_col=col,
        group_col="cluster",
        p_adjust="bonferroni"
    )

    row = {"Atribút": col}
    # uložíme všetky párové porovnania zhlukov do jedného riadku
    for i in range(len(clusters)):
        for j in range(i+1, len(clusters)):

            c1 = clusters[i]
            c2 = clusters[j]

            col_name = f"{c1} vs {c2}"
            row[col_name] = dunn.loc[c1, c2]

    posthoc_results.append(row)

posthoc_table = pd.DataFrame(posthoc_results)

# zaokrúhlenie
for col in posthoc_table.columns[1:]:
    posthoc_table[col] = posthoc_table[col].apply(format_p_value_strict)

posthoc_table

In [ ]:
## Farebná tabuľka

# funkcia prevodu str na číselnú hodnotu
def convert_pvalue_numeric(x):
    if isinstance(x, str):
        x = x.replace(",", ".")
        if "<" in x:
            return float(x.replace("<", ""))  # napr. <0.00001 -> 0.00001
    return float(x)

# funkcia na text (zachová <)
def convert_pvalue_text(x):
    if isinstance(x, str):
        return x.replace(",", ".")
        
    else:
        return f"{x:.3f}"

heatmap_data = posthoc_table.set_index("Atribút")           # pripravíme dáta
numeric_data = heatmap_data.map(convert_pvalue_numeric)     # numerická verzia (pre farby)
text_data = heatmap_data.map(convert_pvalue_text)           # textová verzia (pre zobrazenie)

# vizualizácia
plt.figure(figsize=(6,10))
sns.heatmap(
    numeric_data,
    cmap="coolwarm_r",
    annot=text_data,   # tu dávame text
    fmt="",            
    linewidths=0.5,
    cbar_kws={"label": "Adjusted p-value"}
)

plt.title("Dunn post-hoc test (Bonferroni korekcia)")
plt.xlabel("Porovnanie zhlukov")
plt.ylabel("Atribút")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP hodnoty

# Tréning LightGBM modelu na predikciu klastrov
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score

# X a y pre SHAP
X_shap = X_top36.copy()
y_shap = X_imputed["cluster"].astype(int)

print("Počet vzoriek:", X_shap.shape[0])
print("Počet atribútov:", X_shap.shape[1])
print("Unikátne klastre:", np.sort(y_shap.unique()))

# LightGBM klasifikátor, počet stromov 300, rýchlosť učenia 0,05, vyrovnáva nevyvážené zhluky
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    random_state=42,
    class_weight="balanced"
)

lgbm.fit(X_shap, y_shap)                            # tréning

y_pred = lgbm.predict(X_shap)                       # predikcia
acc = accuracy_score(y_shap, y_pred)                # presnosť 
print(f"Presnosť surrogate modelu (cluster → LGBM): {acc:.3f}")

In [ ]:
# SHAP pokračovanie

import shap

explainer = shap.TreeExplainer(lgbm)                # natrénovaný model
shap_values = explainer.shap_values(X_shap)         # výpis počtu záznamov, atribútov a zhlukov
print("shap_values shape:", np.array(shap_values).shape)

shap_values = np.array(shap_values)
n_samples, n_features, n_classes = shap_values.shape
print(f"Počet vzoriek: {n_samples}")
print(f"Počet atribútov: {n_features}")
print(f"Počet tried (klastrov): {n_classes}")

# Globálna dôležitosť naprieč triedami, spriemerujeme absolútne SHAPy cez triedy
shap_values_global = np.abs(shap_values).mean(axis=2)  # priemer cez triedy → (n_samples, n_features)
shap.summary_plot(
    shap_values_global,
    X_shap, 
    show=False,
    max_display=20,       # zobrazíme top 20
    plot_size=(4, 8)
    )

# vizualizácia
plt.tick_params(axis='y', labelsize=10, color="black")   # ← názvy atribútov
plt.xlabel("SHAP value", fontsize=11)
plt.title("SHAP dôležitosť atribútov k-medoids", fontsize=12, pad=15)
plt.tight_layout()
plt.show()

# samostatné SHAP grafy pre každý cluster
for c in range(n_classes):
    print(f"\n=== SHAP summary pre cluster {c} ===")
    shap.summary_plot(shap_values[:, :, c], X_shap, show=True, max_display=20, plot_size=(5, 8))

In [ ]:
# BAR plot SHAP hodnôt - globálna dôležitosť

shap.summary_plot(
    shap_values_global,
    X_shap,
    plot_type="bar",
    max_display=29,     # možnosť zmeniť počet atribútov
    show=False,
    plot_size = (6, 8),
)

plt.title(f"SHAP Bar Plot – Celková dôležitosť atribútov", fontsize=14)
plt.xlabel("Priemerný absolútny SHAP efekt", fontsize=12)
plt.tight_layout()
plt.show()